## Objective

Build a clean "master dataset" by joining multiple tables and engineering features for analysis and modeling.

Key outputs:
- One row per order (order-level dataset)
- Customer-level features (for churn later)

# Load data

In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv")
orders = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv")
order_items = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv")
payments = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv")
products = pd.read_csv("/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv")

# Convert datetime

In [2]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# Aggregate order_items

In [3]:
order_items_agg = order_items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    total_items=('order_item_id', 'count')
).reset_index()

order_items_agg.head()

,order_id,total_price,total_freight,total_items
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1


# Aggregate payments

In [4]:
payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_types=('payment_type', 'nunique'),
    installments=('payment_installments', 'max')
).reset_index()

# Build master table

In [5]:
df = orders.merge(customers, on='customer_id', how='left')

df = df.merge(order_items_agg, on='order_id', how='left')

df = df.merge(payments_agg, on='order_id', how='left')

df.shape
df.head()

# Feature Engineering

Order value

In [7]:
df['order_value'] = df['total_payment']

Delivery time

In [8]:
df['delivery_time'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

Order year/month

In [9]:
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.month

Customer order count

In [10]:
customer_orders = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'count')
).reset_index()

df = df.merge(customer_orders, on='customer_unique_id', how='left')

Recency

In [11]:
last_purchase = df.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
last_purchase.columns = ['customer_unique_id', 'last_purchase_date']

df = df.merge(last_purchase, on='customer_unique_id', how='left')

df['recency_days'] = (df['order_purchase_timestamp'].max() - df['last_purchase_date']).dt.days

Clean up

In [12]:
df = df[df['order_status'] == 'delivered']

Updated dataset

In [14]:
df.describe()

,order_purchase_timestamp,order_delivered_customer_date,customer_zip_code_prefix,total_price,total_freight,total_items,total_payment,payment_types,installments,order_value,delivery_time,order_year,order_month,total_orders,last_purchase_date,recency_days
count,96478,96470,96478.000000,96478.000000,96478.000000,96478.000000,96477.000000,96477.000000,96477.000000,96477.000000,96470.000000,96478.000000,96478.000000,96478.000000,96478,96478.000000
mean,2018-01-01 23:29:31.939913984,2018-01-14 12:41:33.581683456,35198.185358,137.041586,22.785253,1.142198,159.856357,1.022617,2.928024,159.856357,12.093604,2017.544331,6.031116,1.078401,2018-01-04 21:54:12.711809792,285.350826
min,2016-09-15 12:16:38,2016-10-11 13:46:32,1003.000000,0.850000,0.000000,1.000000,9.590000,1.000000,0.000000,9.590000,0.000000,2016.000000,1.000000,1.000000,2016-09-15 12:16:38,0.000000
25%,2017-09-14 09:00:23.249999872,2017-09-25 22:15:09.500000,11355.000000,45.900000,13.850000,1.000000,61.880000,1.000000,1.000000,61.880000,6.000000,2017.000000,3.000000,1.000000,2017-09-18 19:43:47.500000,162.000000
50%,2018-01-20 19:45:45,2018-02-02 19:32:21,24435.000000,86.575000,17.170000,1.000000,105.280000,1.000000,2.000000,105.280000,10.000000,2018.000000,6.000000,1.000000,2018-01-23 21:05:05.500000,266.000000
75%,2018-05-05 18:54:47,2018-05-15 22:54:48.500000,59056.000000,149.900000,24.017500,1.000000,176.330000,1.000000,4.000000,176.330000,15.000000,2018.000000,8.000000,1.000000,2018-05-07 22:53:40.249999872,393.000000
max,2018-08-29 15:00:37,2018-10-17 13:22:46,99980.000000,13440.000000,1794.960000,21.000000,13664.080000,2.000000,24.000000,13664.080000,209.000000,2018.000000,12.000000,17.000000,2018-10-16 20:16:02,762.000000
std,NaN,NaN,29839.705392,209.045198,21.559197,0.538804,218.813144,0.148679,2.712723,218.813144,9.551380,0.503560,3.228391,0.390940,NaN,152.380521


In [15]:
df.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
total_price                       0
total_freight                     0
total_items                       0
total_payment                     1
payment_types                     1
installments                      1
order_value                       1
delivery_time                     8
order_year                        0
order_month                       0
total_orders                      0
last_purchase_date                0
recency_days                      0
dtype: int64

## Data Processing Summary

- Combined multiple relational tables into a single master dataset
- Aggregated item-level and payment-level data to order-level
- Engineered key features:
  - order_value
  - delivery_time
  - total_orders per customer
  - recency_days

Business implication:
- Dataset is now ready for:
  - customer behavior analysis
  - churn prediction
  - sales analysis

# Output dataset

In [16]:
df.to_csv("processed_master_dataset.csv", index=False)